# Splitting and Augmenting Image Database

Splitting the database in around 70:30 proportion and augmenting considering the exploratory data analysis (EDA)

## Splitting Data

Splitting the data, using the individuals 1, 5, and 10 from all genotypes as tests

In [3]:
import os
from PIL import Image

def split_and_copy_images(base_dir, test_inds=[1, 5, 10]):
    crops = ['wheat', 'sorghum']
    for crop in crops:
        for subfolder in ['images', 'masks']:
            src_folder = os.path.join(base_dir, crop, subfolder)
            train_folder = os.path.join(base_dir, crop, 'train', subfolder)
            test_folder = os.path.join(base_dir, crop, 'test', subfolder)
            os.makedirs(train_folder, exist_ok=True)
            os.makedirs(test_folder, exist_ok=True)
            for fname in os.listdir(src_folder):
                if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
                    continue
                parts = fname.split('_')
                if len(parts) < 3:
                    continue
                try:
                    individual = int(parts[2].split('.')[0])
                except ValueError:
                    continue
                src_path = os.path.join(src_folder, fname)
                if individual in test_inds:
                    dst_path = os.path.join(test_folder, fname)
                else:
                    dst_path = os.path.join(train_folder, fname)
                # Open, resize, and save the image
                with Image.open(src_path) as img:
                    img = img.resize((512, 512), Image.LANCZOS)
                    img.save(dst_path)

In [4]:
split_and_copy_images(base_dir='../data')

In [5]:
def summarize_image_counts(base_dir):
    crops = ['wheat', 'sorghum']
    summary = {}
    for crop in crops:
        summary[crop] = {}
        for split in ['train', 'test']:
            for subfolder in ['images', 'masks']:
                folder = os.path.join(base_dir, crop, split, subfolder)
                count = len([f for f in os.listdir(folder) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'))])
                summary[crop][f'{split}_{subfolder}'] = count
        train_images = summary[crop]['train_images']
        test_images = summary[crop]['test_images']
        proportion = train_images / (train_images + test_images) if (train_images + test_images) > 0 else 0
        print(f"{crop.capitalize()} - Train images: {train_images}, Test images: {test_images}, Proportion (train:test): {proportion:.2f}:{1-proportion:.2f}")
        print(f"{crop.capitalize()} - Train masks: {summary[crop]['train_masks']}, Test masks: {summary[crop]['test_masks']}")
        print('-' * 40)

In [6]:
summarize_image_counts(base_dir='../data')

Wheat - Train images: 1389, Test images: 592, Proportion (train:test): 0.70:0.30
Wheat - Train masks: 1389, Test masks: 592
----------------------------------------
Sorghum - Train images: 2677, Test images: 1148, Proportion (train:test): 0.70:0.30
Sorghum - Train masks: 2677, Test masks: 1148
----------------------------------------


## Augmenting the data

Augmenting be flipping, mirroring, rotating, changing in saturation and brightness.

Making 5 new images per original image.

In [22]:
from PIL import ImageEnhance
import random

import matplotlib.pyplot as plt

def augment_image(img):
    augmented = []
    # 1. Horizontal flip
    augmented.append(img.transpose(Image.FLIP_LEFT_RIGHT))
    # 2. Vertical flip
    augmented.append(img.transpose(Image.FLIP_TOP_BOTTOM))
    # 3. Rotate 90 degrees
    augmented.append(img.rotate(90, expand=True))
    # 4. Change saturation
    enhancer = ImageEnhance.Color(img)
    augmented.append(enhancer.enhance(random.uniform(0.1, 2)))
    # 5. Change brightness
    enhancer = ImageEnhance.Brightness(img)
    augmented.append(enhancer.enhance(random.uniform(0.1, 2)))
    return augmented

def visualize_augmentation(img):
    augmented = augment_image(img)
    fig, axes = plt.subplots(1, len(augmented) + 1, figsize=(15, 4))
    axes[0].imshow(img)
    axes[0].set_title('Original')
    axes[0].axis('off')
    for i, aug_img in enumerate(augmented):
        axes[i+1].imshow(aug_img)
        axes[i+1].set_title(f'Aug {i+1}')
        axes[i+1].axis('off')
    plt.tight_layout()
    plt.show()

def augment_and_save(base_dir, visualize=False):
    crops = ['wheat', 'sorghum']
    for crop in crops:
        for split in ['train', 'test']:
            for subfolder in ['images', 'masks']:
                folder = os.path.join(base_dir, crop, split, subfolder)
                for fname in os.listdir(folder):
                    if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
                        continue
                    fpath = os.path.join(folder, fname)
                    with Image.open(fpath) as img:
                        # For masks, only apply geometric transforms (no color/brightness)
                        if subfolder == 'masks':
                            # For masks, only apply geometric transforms (flips and 90-degree rotation), 
                            # for Aug 4 and 5 just copy the original mask (no-op)
                            augmented = [
                                img.transpose(Image.FLIP_LEFT_RIGHT),
                                img.transpose(Image.FLIP_TOP_BOTTOM),
                                img.rotate(90, expand=True),
                                img.copy(),  # Aug 4: just copy the original mask
                                img.copy()   # Aug 5: just copy the original mask
                            ]
                            if visualize:
                                fig, axes = plt.subplots(1, len(augmented) + 1, figsize=(15, 4))
                                axes[0].imshow(img)
                                axes[0].set_title('Original Mask')
                                axes[0].axis('off')
                                for i, aug_img in enumerate(augmented):
                                    axes[i+1].imshow(aug_img)
                                    axes[i+1].set_title(f'Aug {i+1}')
                                    axes[i+1].axis('off')
                                plt.tight_layout()
                                plt.show()
                                # Only visualize the first mask and return
                                return
                        else:
                            augmented = augment_image(img)
                            if visualize:
                                visualize_augmentation(img)
                                # Only visualize the first image and return
                                return
                        for i, aug_img in enumerate(augmented):
                            name, ext = os.path.splitext(fname)
                            aug_fname = f"{name}_aug{i+1}{ext}"
                            aug_img.save(os.path.join(folder, aug_fname))

In [24]:
augment_and_save(base_dir='../data', visualize=False)